# 07: Alignment Basics

This tutorial introduces alignment - the core capability that connects timelines and enables coordinate transfer between them.

**Learning Objectives:**
- Understand alignment as establishing commensurability between timelines
- Use `TimelineGroup` to manage commensurable timelines
- Define alignments with `PerfectAlignment`
- Transfer coordinates between timelines in a group

**Prerequisites:**
- Notebooks 01-06 (Core Concepts, Loaders, C-Maps, Timelines, Timestamps, Graphical Timelines)
- Understanding of Timeline objects and Coordinate types

## Part 1: What is Alignment?

In previous tutorials, we learned:
- **Timestamps** (Notebook 05): Cross-section views through timeline hierarchies - they tell us WHERE events are
- **Graphical Timelines** (Notebook 06): Mapping images to timelines

But timestamps are **read-only**. They don't let us convert coordinates from one timeline to another.

**Alignment** establishes **commensurability** between timelines - the ability to convert coordinates from one to another:
> "If I'm at coordinate X in Timeline A, what's the corresponding coordinate in Timeline B?"

### Use Cases
- **Score-Audio Sync**: "At measure 5 in the score, what's the timestamp in the audio?"
- **Cross-Version Comparison**: "This pixel in Analysis A corresponds to which pixel in Analysis B?"
- **OMR Linking**: "This note in the scanned image maps to which MIDI event?"

## Part 2: Commensurability

Two timelines are **commensurable** if there exists a way to convert coordinates between them.

In TimeToAlign!, commensurability is established by placing timelines in a **TimelineGroup** with a shared reference. Any timeline in the group can be converted to any other via the reference.

```
Timeline A  <-->  Reference  <-->  Timeline B
   (MIDI)         (pixels)         (audio)
```

### PerfectAlignment

A `PerfectAlignment` defines a **linear mapping** between a timeline and the reference:

```
Source:     [source_start ----------- source_end]
                    |                     |
                    v  (linear mapping)   v
                    |                     |
Reference:  [ref_start --------------- ref_end]
```

This is the simplest form of alignment - if you know the boundary correspondence, you can interpolate any point in between.

## Setup

In [ ]:
from pathlib import Path

from timetoalign.alignment import PerfectAlignment, TimelineGroup
from timetoalign.timelines import (
    ContinuousPhysicalTimeline,
    Timeline,
)

# For the Thoresen example
from timetoalign.loader.graphical import GraphicalLoader

## Part 3: TimelineGroup

A `TimelineGroup` is a collection of **commensurable timelines** - timelines that can be converted to each other through a shared reference.

### Creating a Group with a Reference Timeline

In [ ]:
# Create an image timeline (e.g., a piano roll image)
image_timeline = Timeline(
    length=10000,  # 10,000 pixels tall
    uid="image",
    name="Piano Roll Image",
)

# Create a group with this timeline as the reference
group = TimelineGroup.from_reference(image_timeline, name="Piano Roll Group")

{
    "group_id": group.id,
    "name": group.name,
    "reference": group.reference_timeline_id,
    "n_timelines": group.n_timelines,
}

### Adding Timelines to the Group

When you add a timeline to the group, you specify how it maps to the reference using a `PerfectAlignment`.

In [ ]:
# Create an audio timeline
audio_timeline = ContinuousPhysicalTimeline(
    length=300.0,  # 5 minutes = 300 seconds
    unit="seconds",
    uid="audio",
    name="Audio Recording",
)

# Add to group with full-extent linear alignment
# [0, 300] seconds maps to [0, 10000] pixels
group.add_timeline(
    audio_timeline,
    alignment=PerfectAlignment(
        source_start=0,
        source_end=300.0,
        ref_start=0,
        ref_end=10000,
    ),
)

{
    "n_timelines": group.n_timelines,
    "timelines": [tid for tid, _, _ in group.iter_timelines()],
}

### Partial Alignment

Timelines don't have to cover the full extent of the reference. Specify the exact mapping:

In [ ]:
# Create a MIDI timeline
midi_timeline = Timeline(
    length=87180,  # MIDI ticks
    uid="midi",
    name="MIDI File",
)

# The MIDI only covers the middle portion of the image
# MIDI ticks [0, 87180] map to image pixels [1000, 9000]
group.add_timeline(
    midi_timeline,
    alignment=PerfectAlignment(
        source_start=0,
        source_end=87180,
        ref_start=1000,  # Music starts at pixel 1000
        ref_end=9000,    # Music ends at pixel 9000
    ),
)

group.n_timelines

### Coordinate Transfer

Now that all three timelines are commensurable (in the same group), we can convert between any pair:

In [ ]:
# Transfer from audio seconds to image pixels
audio_coord = 150.0  # 2.5 minutes
pixel_coord = group.convert(audio_coord, from_timeline="audio", to_timeline="image")

print(f"Audio: {audio_coord} seconds -> Image: {pixel_coord:.1f} pixels")

# Transfer from MIDI ticks to audio seconds
midi_coord = 43590  # Halfway through the MIDI
audio_from_midi = group.convert(midi_coord, from_timeline="midi", to_timeline="audio")

print(f"MIDI: {midi_coord} ticks -> Audio: {audio_from_midi:.2f} seconds")

### How Conversion Works

All conversions go through the reference timeline:

```
MIDI (source)  -->  Reference (pixels)  -->  Audio (target)
   43590       -->       5000           -->      150.0
```

The group's reference is the common coordinate system that makes all member timelines commensurable.

### Group Summary

In [ ]:
import json
print(json.dumps(group.summary(), indent=2))

---

## Part 4: Real-World Example - Thoresen Cross-Version Analysis

Let's apply these concepts to real data: two different graphical analyses of the same music.

### The Data
- **DGT1 (2009)**: Single image with 5 horizontal systems (4835 pixels total)
- **DGT2 (2010)**: 5 separate images (4328 pixels total)
- Both represent the same **150-second audio excerpt**

We want to make coordinates transferable between DGT1 and DGT2. Since both analyses represent the same audio, we can establish commensurability through a shared audio timeline.

In [ ]:
# Path to test data (adjust as needed)
data_dir = Path("../../tests/alignment/data/thoresen")

# DGT1: Single image with 5 systems
dgt1_image = data_dir / "thoresen_2009_sound-objects_p312_page1_1.jpeg"

# DGT1 segment parameters
DGT1_X0, DGT1_X1 = 2, 969
DGT1_Y_POSITIONS = [18, 205, 396, 588, 785]

# Load DGT1
loader1 = GraphicalLoader(metadata={"source": "Thoresen 2009"})
idx1 = loader1.add_image(dgt1_image)

for i, y in enumerate(DGT1_Y_POSITIONS):
    loader1.add_horizontal_segment(
        source_index=idx1,
        x0=DGT1_X0,
        x1=DGT1_X1,
        y=y,
        name=f"system_{i+1}",
    )

dgt1_bundle = loader1.bundle
print(f"DGT1: {dgt1_bundle.n_segments} segments, {dgt1_bundle.total_length:.0f} pixels")

In [ ]:
# DGT2: 5 separate images
dgt2_images = [
    data_dir / "thoresen_2010_form-building-patterns_p90-91_page1_1.jpeg",
    data_dir / "thoresen_2010_form-building-patterns_p90-91_page1_2.jpeg",
    data_dir / "thoresen_2010_form-building-patterns_p90-91_page1_3.jpeg",
    data_dir / "thoresen_2010_form-building-patterns_p90-91_page1_4.jpeg",
    data_dir / "thoresen_2010_form-building-patterns_p90-91_page2_1.jpeg",
]

DGT2_SEGMENT_BOUNDS = [
    (8, 874, 15),
    (7, 874, 18),
    (7, 874, 19),
    (8, 872, 15),
    (9, 873, 20),
]

loader2 = GraphicalLoader(metadata={"source": "Thoresen 2010"})

for i, (img_path, (x0, x1, y)) in enumerate(zip(dgt2_images, DGT2_SEGMENT_BOUNDS)):
    idx2 = loader2.add_image(img_path)
    loader2.add_horizontal_segment(
        source_index=idx2,
        x0=x0,
        x1=x1,
        y=y,
        name=f"page_{i+1}",
    )

dgt2_bundle = loader2.bundle
print(f"DGT2: {dgt2_bundle.n_segments} segments, {dgt2_bundle.total_length:.0f} pixels")

### Create Timelines

In [ ]:
# Convert bundles to timelines
dgt1_timeline = dgt1_bundle.to_timeline(uid="dgt1", name="Thoresen 2009")
dgt2_timeline = dgt2_bundle.to_timeline(uid="dgt2", name="Thoresen 2010")

# Create audio timeline (150 seconds)
AUDIO_DURATION = 150.0
audio_timeline = ContinuousPhysicalTimeline(
    length=AUDIO_DURATION,
    unit="seconds",
    uid="audio",
    name="Audio",
)

{
    "DGT1": f"{dgt1_timeline.length.value:.0f} pixels",
    "DGT2": f"{dgt2_timeline.length.value:.0f} pixels",
    "Audio": f"{audio_timeline.length.value:.1f} seconds",
}

### Create TimelineGroups

We create two groups, each establishing commensurability between a graphical timeline and the audio:
- **Group 1**: DGT1 (reference) + Audio
- **Group 2**: DGT2 (reference) + Audio

Since both groups include the same audio, we can transfer coordinates from DGT1 to DGT2 by going through audio.

In [ ]:
# Group 1: DGT1 as reference, audio commensurable with it
dgt1_group = TimelineGroup.from_reference(dgt1_timeline, name="DGT1_Group")
dgt1_group.add_timeline(
    audio_timeline,
    alignment=PerfectAlignment(
        source_start=0,
        source_end=AUDIO_DURATION,
        ref_start=0,
        ref_end=dgt1_timeline.length.value,
    ),
)

# Group 2: DGT2 as reference, audio commensurable with it
dgt2_group = TimelineGroup.from_reference(dgt2_timeline, name="DGT2_Group")
dgt2_group.add_timeline(
    audio_timeline,
    alignment=PerfectAlignment(
        source_start=0,
        source_end=AUDIO_DURATION,
        ref_start=0,
        ref_end=dgt2_timeline.length.value,
    ),
)

{
    "DGT1 Group": dgt1_group.n_timelines,
    "DGT2 Group": dgt2_group.n_timelines,
}

### Coordinate Transfer: DGT2 -> DGT1

To transfer a coordinate from DGT2 to DGT1, we go through the shared audio timeline:

```
DGT2 pixels  -->  Audio seconds  -->  DGT1 pixels
  (Group 2)                           (Group 1)
```

In [ ]:
# Test pixel in DGT2 (halfway through)
dgt2_pixel = 2164.0

# Step 1: DGT2 -> Audio (via DGT2 group)
audio_seconds = dgt2_group.convert(dgt2_pixel, from_timeline="dgt2", to_timeline="audio")
print(f"DGT2 pixel {dgt2_pixel:.0f} -> Audio {audio_seconds:.2f} seconds")

# Step 2: Audio -> DGT1 (via DGT1 group)
dgt1_pixel = dgt1_group.convert(audio_seconds, from_timeline="audio", to_timeline="dgt1")
print(f"Audio {audio_seconds:.2f} seconds -> DGT1 pixel {dgt1_pixel:.1f}")

# Verify: Both should be at 50% (75 seconds)
print(f"\nVerification: {audio_seconds / AUDIO_DURATION * 100:.1f}% through the piece")

---

## Part 5: AlignmentBundle (Optional)

For projects with many timelines across multiple groups, `AlignmentBundle` provides a convenience wrapper that manages groups and provides a unified interface.

Most use cases are well-served by `TimelineGroup` directly. Consider `AlignmentBundle` when you need:
- Centralized management of many timelines across multiple groups
- User-friendly UID aliases separate from timeline IDs
- Future cross-group transfer capabilities (Phase 2+)

In [ ]:
from timetoalign.alignment import AlignmentBundle

# AlignmentBundle wraps multiple groups
bundle = AlignmentBundle(name="Thoresen Project")

# Add timelines - bundle creates groups automatically
bundle.add_timeline(dgt1_timeline, uid="dgt1")
bundle.add_timeline(
    audio_timeline,
    uid="audio_1",
    aligned_to="dgt1",
    alignment=PerfectAlignment(
        source_start=0,
        source_end=AUDIO_DURATION,
        ref_start=0,
        ref_end=dgt1_timeline.length.value,
    ),
)

# Transfer works the same way
result = bundle.transfer(150.0, from_timeline="audio_1", to_timeline="dgt1")
print(f"Audio 150s -> DGT1 {result:.1f} pixels")

---

## Part 6: Roadmap

### What's Implemented (Phase 1)
- `TimelineGroup` for managing commensurable timelines
- `PerfectAlignment` for linear coordinate mapping
- `AlignmentBundle` for multi-group management

### Coming Soon (Phase 2+)
- **MatchClaim**: Event-to-event correspondence
- **MatchGraph**: Network of match claims
- **WarpMap**: Non-linear alignment derived from match data
- **Cross-group transfer** via shared timelines (automated)

### Next Tutorial
See **Application A3: SUPRA Piano Roll** for an alignment workflow with IIIF image metadata and ATON hole punch data.

---

## Summary

**Key Takeaways:**

1. **Commensurability** is the ability to convert coordinates between timelines
2. **TimelineGroup** establishes commensurability through a shared reference
3. **PerfectAlignment** defines the linear mapping between a timeline and the reference
4. **convert()** transfers coordinates between any commensurable timelines in the group

> "A TimelineGroup establishes commensurability - it connects timelines through a common reference, enabling coordinate transfer between any pair in the group."

---

## Exercises

### Exercise 1: Basic Group
Create a TimelineGroup with:
- A score timeline (1000 quarters) as reference
- A performance timeline (240 seconds) aligned to it

Transfer the coordinate for measure 50 (assuming 4 quarters per measure) to performance seconds.

<details>
<summary>Solution</summary>

```python
score = Timeline(length=1000, uid="score")
perf = ContinuousPhysicalTimeline(length=240.0, unit="seconds", uid="perf")

group = TimelineGroup.from_reference(score, name="Score-Perf")
group.add_timeline(
    perf,
    alignment=PerfectAlignment(
        source_start=0,
        source_end=240.0,
        ref_start=0,
        ref_end=1000,
    ),
)

measure_50 = 50 * 4  # 200 quarters
perf_seconds = group.convert(measure_50, "score", "perf")
print(f"Measure 50 -> {perf_seconds:.1f} seconds")
# Expected: 48.0 seconds (200/1000 * 240)
```
</details>

### Exercise 2: Partial Alignment
Create a group where a MIDI file (0-48000 ticks) only covers measures 10-50 of a score (1000 quarters).

<details>
<summary>Solution</summary>

```python
score = Timeline(length=1000, uid="score")
midi = Timeline(length=48000, uid="midi")

group = TimelineGroup.from_reference(score, name="Score-MIDI")
group.add_timeline(
    midi,
    alignment=PerfectAlignment(
        source_start=0,
        source_end=48000,
        ref_start=40,    # Measure 10 = 40 quarters
        ref_end=200,     # Measure 50 = 200 quarters
    ),
)

# MIDI tick 24000 (halfway) should map to measure 30
score_quarters = group.convert(24000, "midi", "score")
print(f"MIDI 24000 -> {score_quarters:.0f} quarters (measure {score_quarters/4:.0f})")
# Expected: 120 quarters (measure 30)
```
</details>